# 02_04 · Preprocesado — SPARK TEC Electrical Monitoring

**Entrada**: `data/processed/spark_tec_1min.parquet` — generado por `01_04_EDA_SPARK`
**Salida**: `spark_{MACHINE}_1min.parquet` por máquina + `spark_scalers.pkl`

**Origen**: SPARK dataset — FIZ Karlsruhe / Leibniz-Institut  
**DOI**: [10.35097/bjdg3m3rg5jv3skk](https://doi.org/10.35097/bjdg3m3rg5jv3skk)

Pipeline:
1. Carga del dataset agregado a 1 minuto (desde EDA)
2. Detección y marcado de gaps (standby: `P_total_mean` < 50 W)
3. Escalado por máquina (StandardScaler ajustado solo sobre minutos en operación)

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

BASE      = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
PROCESSED = os.path.join(BASE, 'data', 'processed')

MACHINES = sorted([
    'TEC_48S', 'TEC_CFST161', 'TEC_CTX800TC', 'TEC_Chiron800',
    'TEC_DMF3008', 'TEC_DMU125MB', 'TEC_DNG50evo', 'TEC_E110',
    'TEC_E30D2', 'TEC_JWA24', 'TEC_MV2400R'
])
VARS = ['P_total','P1','P2','P3','I1','I2','I3','Freq','THD_I1','THD_I2','THD_I3','PF_total']
FEATURE_COLS = [f'{v}_{s}' for v in VARS for s in ['mean','std','max']]
GAP_W = 50

print(f'Features: {len(FEATURE_COLS)} | Máquinas: {len(MACHINES)}')


Features: 36 | Máquinas: 11


## 1. Carga del dataset agregado

Leemos el parquet generado por `01_04_EDA_SPARK` (1 min, sin escalar, sin gaps marcados).


In [2]:
df_all = pd.read_parquet(os.path.join(PROCESSED, 'spark_tec_1min.parquet'))

print(f'Shape: {df_all.shape}')
print(f'Periodo: {df_all.index.min()} → {df_all.index.max()}')
print(f'Máquinas: {sorted(df_all["machine"].unique())}')
df_all.head(3)


Shape: (5797440, 37)
Periodo: 2024-01-01 00:00:00 → 2024-12-31 23:59:00
Máquinas: ['TEC_48S', 'TEC_CFST161', 'TEC_CTX800TC', 'TEC_Chiron800', 'TEC_DMF3008', 'TEC_DMU125MB', 'TEC_DNG50evo', 'TEC_E110', 'TEC_E30D2', 'TEC_JWA24', 'TEC_MV2400R']


,P_total_mean,P_total_std,P_total_max,P1_mean,P1_std,P1_max,P2_mean,P2_std,P2_max,P3_mean,...,THD_I2_mean,THD_I2_std,THD_I2_max,THD_I3_mean,THD_I3_std,THD_I3_max,PF_total_mean,PF_total_std,PF_total_max,machine
WsDateTime,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,TEC_48S
2024-01-01 00:01:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,TEC_48S
2024-01-01 00:02:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,TEC_48S


## 2. Detección y marcado de gaps

Un gap es cualquier minuto donde la máquina está en standby o apagada: `|P_total_mean| < 50 W`.
Los gaps se marcan con `is_gap=True` y se excluyen del entrenamiento del modelo.
Se guarda un parquet por máquina para que los notebooks de modelado puedan cargar cada una individualmente.


In [3]:
for m in MACHINES:
    df_m = df_all[df_all['machine'] == m].copy()
    df_m['is_gap'] = df_m['P_total_mean'].abs() < GAP_W
    df_m.to_parquet(os.path.join(PROCESSED, f'spark_{m}_1min.parquet'))
    n_gap = df_m['is_gap'].sum()
    print(f'{m:<22} {len(df_m):>7,} min  gaps={n_gap:>6,} ({n_gap/len(df_m)*100:.1f}%)')


TEC_48S                527,040 min  gaps=465,680 (88.4%)
TEC_CFST161            527,040 min  gaps=    12 (0.0%)
TEC_CTX800TC           527,040 min  gaps=412,034 (78.2%)
TEC_Chiron800          527,040 min  gaps=386,198 (73.3%)
TEC_DMF3008            527,040 min  gaps=390,362 (74.1%)
TEC_DMU125MB           527,040 min  gaps=423,539 (80.4%)
TEC_DNG50evo           527,040 min  gaps=     1 (0.0%)
TEC_E110               527,040 min  gaps=395,589 (75.1%)
TEC_E30D2              527,040 min  gaps=411,830 (78.1%)
TEC_JWA24              527,040 min  gaps=     2 (0.0%)
TEC_MV2400R            527,040 min  gaps=296,231 (56.2%)


## 3. Escalado por máquina — StandardScaler

Isolation Forest mide distancias implícitamente. Si las features tienen escalas muy distintas
(P_total en kW vs Freq en Hz), las de mayor escala dominan la partición.

El scaler se ajusta **solo sobre minutos en operación** (excluyendo gaps) para no distorsionar
la distribución con valores de standby.


In [4]:
scalers = {}
for m in MACHINES:
    df_m = pd.read_parquet(os.path.join(PROCESSED, f'spark_{m}_1min.parquet'))
    feat_cols = [c for c in FEATURE_COLS if c in df_m.columns]
    df_on = df_m[~df_m['is_gap']][feat_cols].dropna()
    scalers[m] = StandardScaler().fit(df_on)
    print(f'{m:<22} escalado sobre {len(df_on):,} minutos en operación')

with open(os.path.join(PROCESSED, 'spark_scalers.pkl'), 'wb') as f:
    pickle.dump(scalers, f)
print('\nScalers guardados OK')

TEC_48S                escalado sobre 15,802 minutos en operación
TEC_CFST161            escalado sobre 477,781 minutos en operación
TEC_CTX800TC           escalado sobre 65,787 minutos en operación
TEC_Chiron800          escalado sobre 104,596 minutos en operación
TEC_DMF3008            escalado sobre 89,196 minutos en operación
TEC_DMU125MB           escalado sobre 73,744 minutos en operación
TEC_DNG50evo           escalado sobre 477,786 minutos en operación
TEC_E110               escalado sobre 83,161 minutos en operación
TEC_E30D2              escalado sobre 85,165 minutos en operación
TEC_JWA24              escalado sobre 477,788 minutos en operación
TEC_MV2400R            escalado sobre 181,588 minutos en operación

Scalers guardados OK


## 4. Conclusiones — Preprocesado SPARK

- **36 features** (mean/std/max × 12 señales) capturan nivel, variabilidad y picos de cada señal eléctrica
- **Gaps identificados**: entre el 5% y el 88% de los registros por máquina corresponden a standby (varía mucho según la máquina)
- **Escalado individual**: cada máquina tiene su propio scaler ajustado sobre su distribución operativa
- **Output**: 11 parquets individuales + `spark_scalers.pkl` listos para el modelo